# 🚗 UA-DETRAC Trafik Veri Seti ile YOLO11 Araç Tespiti (Uçtan Uca Pipeline)
### Bitirme Projesi - Gerçek Trafik Kameraları ile Yüksek Başarımlı Model Eğitimi

Bu notebook, Google Drive'ınızdaki **`DETRAC-Images.zip`** arşivini otomatik olarak bulur, etiketleri işler, **YOLO11** modelini T4 GPU ile eğitir ve sonuçları paketler.

👉 **Tüm adımları yukarıdan aşağıya sırayla (1'den 7'ye) çalıştırmanız yeterlidir.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## Adım 1: Donanım Kontrolü ve Kütüphane Kurulumu
GPU hızlandırıcısının (T4) devrede olduğunu doğrulayıp Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [ ]:
import os
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! T4 GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Lütfen üst menüden:')
    print('Çalışma zamanı (Runtime) -> Çalışma zamanı türünü değiştir (Change runtime type) -> T4 GPU seçip Kaydet deyin.')

!pip install -q ultralytics
print('✅ Ultralytics ve PyTorch ortamı hazır!')


## Adım 2: Proje Deposunu ve XML Etiketlerini Çekme
GitHub üzerindeki repomuzu ve içerisinde hazır bulunan XML koordinat etiketlerini çekiyoruz.


In [ ]:
import os

repo_dir = '/content/federated-traffic-learning'
if not os.path.exists(repo_dir):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git /content/federated-traffic-learning
    %cd /content/federated-traffic-learning
else:
    %cd /content/federated-traffic-learning
    !git pull

print('✅ Proje dizini ve XML etiket paketi hazır:', os.getcwd())


## Adım 3: Google Drive'daki DETRAC-Images.zip'i Bulma ve YOLO Formatına Dönüştürme
Drive'a attığınız **`DETRAC-Images.zip`** dosyasını büyük/küçük harf duyarsız olarak bulur, repodaki XML etiketleriyle eşleştirip doğrudan YOLO formatına dönüştürür.


In [ ]:
import os
import glob
import zipfile
import random
import xml.etree.ElementTree as ET
from google.colab import drive

# 1. Google Drive'ı bağla
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Büyük/Küçük harf duyarsız esnek zip arama (Drive ve /content)
print('🔍 Google Drive ve yerel dizinde DETRAC zip dosyası aranıyor...')
found_zips = []

# Önce /content dizinini kontrol et
for f in os.listdir('/content'):
    if f.endswith('.zip') and ('detrac' in f.lower() or 'vehicle' in f.lower()):
        found_zips.append(os.path.join('/content', f))

# Drive içerisinde os.walk ile büyük/küçük harf duyarsız arama
if not found_zips and os.path.exists('/content/drive/MyDrive'):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if '.trash' in root.lower():
            continue
        for f in files:
            if f.endswith('.zip') and ('detrac' in f.lower() or 'vehicle' in f.lower()):
                found_zips.append(os.path.join(root, f))
        if found_zips:
            break

# Eğer hala bulunamadıysa Drive'daki tüm zip dosyalarını tara
if not found_zips and os.path.exists('/content/drive/MyDrive'):
    all_zips = []
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if '.trash' in root.lower():
            continue
        for f in files:
            if f.endswith('.zip'):
                all_zips.append(os.path.join(root, f))
    if all_zips:
        print('ℹ️ Drive içinde bulunan mevcut zip dosyaları:')
        for az in all_zips:
            print(' -', az)
        found_zips = all_zips

if not found_zips:
    raise FileNotFoundError('❌ Google Drive içerisinde DETRAC-Images.zip dosyası bulunamadı! Lütfen dosyanın Drive ana dizininde olduğundan emin olun.')

images_zip_path = found_zips[0]
size_mb = os.path.getsize(images_zip_path) / (1024 * 1024)
print(f'✅ Veri seti arşivi bulundu: {images_zip_path} ({size_mb:.2f} MB)')

# 3. Hedef dizinleri hazırla
out_dataset_dir = '/content/ua_detrac_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(out_dataset_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(out_dataset_dir, 'labels', split), exist_ok=True)

# Arşiv içeriğini kontrol et (Ham DETRAC mı yoksa önceden hazırlanmış YOLO verisi mi?)
with zipfile.ZipFile(images_zip_path, 'r') as zf_check:
    first_entries = zf_check.namelist()[:30]
    is_raw_detrac = any('MVI_' in e for e in first_entries)
    is_yolo_ready = any('train' in e.lower() or 'images' in e.lower() for e in first_entries)

if is_raw_detrac:
    print('🎬 Ham DETRAC video kareleri tespit edildi. XML etiketleri ile eşleştirilip YOLO formatına dönüştürülüyor...')
    
    # XML etiket arşivini aç
    xml_zip_path = '/content/federated-traffic-learning/detrac_xml_annotations.zip'
    annot_dir = '/content/DETRAC_annotations'
    if not os.path.exists(annot_dir):
        print('📦 XML etiketleri açılıyor...')
        !unzip -q -o "{xml_zip_path}" -d "{annot_dir}"
        print('✅ XML etiketleri hazır!')
    
    CLASS_MAP = {'car': 0, 'van': 0, 'bus': 2, 'others': 3}
    IMG_W, IMG_H = 960.0, 540.0
    
    def parse_xml(xml_p):
        tree = ET.parse(xml_p)
        root = tree.getroot()
        seq = root.attrib.get('name', os.path.splitext(os.path.basename(xml_p))[0]).strip()
        frames = {}
        for fe in root.iter('frame'):
            n = fe.attrib.get('num')
            if not n:
                continue
            fnum = int(n)
            boxes = []
            tl = fe.find('target_list')
            if tl is not None:
                for tgt in tl.findall('target'):
                    bx = tgt.find('box')
                    att = tgt.find('attribute')
                    if bx is None or att is None:
                        continue
                    vt = att.attrib.get('vehicle_type', 'car').lower()
                    if vt not in CLASS_MAP:
                        continue
                    cid = CLASS_MAP[vt]
                    try:
                        l = float(bx.attrib.get('left', 0))
                        t = float(bx.attrib.get('top', 0))
                        w = float(bx.attrib.get('width', 0))
                        h = float(bx.attrib.get('height', 0))
                    except:
                        continue
                    if w < 5 or h < 5:
                        continue
                    xc = max(0.0, min(1.0, (l + w / 2.0) / IMG_W))
                    yc = max(0.0, min(1.0, (t + h / 2.0) / IMG_H))
                    wn = max(0.001, min(1.0, w / IMG_W))
                    hn = max(0.001, min(1.0, h / IMG_H))
                    boxes.append((cid, xc, yc, wn, hn))
            frames[fnum] = boxes
        return seq, frames
    
    train_xml_dir = os.path.join(annot_dir, 'train')
    test_xml_dir = os.path.join(annot_dir, 'test')
    train_xml_files = sorted(glob.glob(f'{train_xml_dir}/*.xml'))
    test_xml_files = sorted(glob.glob(f'{test_xml_dir}/*.xml'))
    
    random.seed(42)
    random.shuffle(train_xml_files)
    val_count = max(6, int(len(train_xml_files) * 0.15))
    val_xmls = train_xml_files[:val_count]
    train_xmls = train_xml_files[val_count:]
    
    print(f'🎬 Video Dizileri -> Eğitim: {len(train_xmls)}, Doğrulama: {len(val_xmls)}, Test: {len(test_xml_files)}')
    print('🚀 En kaliteli sahneler örneklenip etiketleniyor (1-2 dakika sürer)...')
    
    zf = zipfile.ZipFile(images_zip_path, 'r')
    zip_names = set(zf.namelist())
    
    def extract_split(xml_list, split_name, step):
        img_dir = os.path.join(out_dataset_dir, 'images', split_name)
        lbl_dir = os.path.join(out_dataset_dir, 'labels', split_name)
        count = 0
        for xp in xml_list:
            seq_name, f_dict = parse_xml(xp)
            for fnum in sorted(f_dict.keys())[::step]:
                boxes = f_dict[fnum]
                if not boxes:
                    continue
                key = f'DETRAC-Images/{seq_name}/img{fnum:05d}.jpg'
                if key not in zip_names:
                    key = f'{seq_name}/img{fnum:05d}.jpg'
                    if key not in zip_names:
                        continue
                fname = f'{seq_name}_img{fnum:05d}'
                with open(os.path.join(img_dir, f'{fname}.jpg'), 'wb') as img_f:
                    img_f.write(zf.read(key))
                with open(os.path.join(lbl_dir, f'{fname}.txt'), 'w', encoding='utf-8') as lbl_f:
                    for b in boxes:
                        lbl_f.write(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')
                count += 1
        return count
    
    train_cnt = extract_split(train_xmls, 'train', step=20)
    val_cnt = extract_split(val_xmls, 'val', step=20)
    test_cnt = extract_split(test_xml_files, 'test', step=30)
    zf.close()
    
else:
    print('📦 Hazır YOLO veri seti tespit edildi. Doğrudan açılıyor...')
    !unzip -q -o "{images_zip_path}" -d "{out_dataset_dir}"

print('\n' + '=' * 60)
print('✅ ADIM 3 BAŞARIYLA TAMAMLANDI!')
print(f' • Hedef Dizin: {out_dataset_dir}')
print('=' * 60)


## Adım 4: Veri Seti Detaylı Analiz Raporu ve data.yaml Yapılandırması 📊
Hazırlanan veri setindeki tüm araç sınıflarını (`car`, `motorcycle`, `bus`, `truck`), etiketli araç sayılarını ve oranlarını ekrana döker, `data.yaml` dosyasını oluşturur.


In [ ]:
import os
import glob
from collections import Counter

# Kök dizini dinamik olarak tespit et
dataset_root = '/content/ua_detrac_dataset'
for pr in ['/content/ua_detrac_dataset', '/content/ua_detrac_dataset/vehicle-detection-clean']:
    if os.path.exists(os.path.join(pr, 'images/train')) or os.path.exists(os.path.join(pr, 'train/images')):
        dataset_root = pr
        break

# Klasör yapısını belirle (images/train mi yoksa train/images mi?)
if os.path.exists(os.path.join(dataset_root, 'images/train')):
    train_rel, val_rel, test_rel = 'images/train', 'images/val', 'images/test'
    train_lbl_rel, val_lbl_rel, test_lbl_rel = 'labels/train', 'labels/val', 'labels/test'
else:
    train_rel = 'train/images'
    val_rel = 'val/images' if os.path.exists(os.path.join(dataset_root, 'val/images')) else 'train/images'
    test_rel = 'test/images' if os.path.exists(os.path.join(dataset_root, 'test/images')) else val_rel
    train_lbl_rel = 'train/labels'
    val_lbl_rel = 'val/labels' if os.path.exists(os.path.join(dataset_root, 'val/labels')) else 'train/labels'
    test_lbl_rel = 'test/labels' if os.path.exists(os.path.join(dataset_root, 'test/labels')) else val_lbl_rel

# 1. data.yaml dosyasını oluştur
yaml_text = f"""# Vehicle Detection YOLO Dataset Configuration
path: {dataset_root}
train: {train_rel}
val: {val_rel}
test: {test_rel}

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""
data_yaml_path = '/content/ua_detrac_dataset/data.yaml'
with open(data_yaml_path, 'w', encoding='utf-8') as yf:
    yf.write(yaml_text.strip() + '\n')

# 2. Detaylı Sınıf ve Kutu Dağılımını Analiz Et
class_names_map = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}
print('=' * 68)
print('         📊 YENİ TRAFİK VERİ SETİ DETAYLI ANALİZ RAPORU')
print('=' * 68)
print(f'📁 Veri Seti Dizin Yolu: {dataset_root}\n')

total_overall_boxes = 0
overall_class_counts = Counter()

splits_info = [
    ('train', os.path.join(dataset_root, train_rel), os.path.join(dataset_root, train_lbl_rel)),
    ('val', os.path.join(dataset_root, val_rel), os.path.join(dataset_root, val_lbl_rel)),
    ('test', os.path.join(dataset_root, test_rel), os.path.join(dataset_root, test_lbl_rel))
]

for split_name, img_dir, lbl_dir in splits_info:
    if not os.path.exists(img_dir):
        continue
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    lbls = glob.glob(f'{lbl_dir}/*.txt')
    
    class_counts = Counter()
    split_boxes = 0
    for lf in lbls:
        try:
            with open(lf, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = int(float(parts[0]))
                        class_counts[cls_id] += 1
                        split_boxes += 1
        except:
            pass
            
    overall_class_counts.update(class_counts)
    total_overall_boxes += split_boxes
    
    print(f'📌 [{split_name.upper()} KÜMESİ]')
    print(f'   • Toplam Görsel Sayısı: {len(imgs):,}')
    print(f'   • Toplam Araç Kutusu (Bounding Box): {split_boxes:,}')
    print('   • Sınıf Dağılımı:')
    for cid in sorted(class_counts.keys()):
        cname = class_names_map.get(cid, f'class_{cid}')
        pct = (class_counts[cid] / split_boxes * 100) if split_boxes > 0 else 0.0
        print(f'      - ID {cid:02d} ({cname:<12}): {class_counts[cid]:>7,} adet  (%{pct:>5.1f})')
    print('-' * 68)

print('\n🎯 [GENEL TOPLAM]')
print(f'   • Toplam Etiketlenen Araç Sayısı: {total_overall_boxes:,}')
for cid in sorted(overall_class_counts.keys()):
    cname = class_names_map.get(cid, f'class_{cid}')
    pct = (overall_class_counts[cid] / total_overall_boxes * 100) if total_overall_boxes > 0 else 0.0
    print(f'   • Sınıf {cid} ({cname:<12}): {overall_class_counts[cid]:>8,} adet  (%{pct:>5.1f})')
print('=' * 68)
print(f'✅ data.yaml hazır: {data_yaml_path}')


## Adım 5: YOLO11 Modelini Eğitme (Training) 🚀
Hazırlanan veri seti üzerinde **YOLO11 Nano** modelini T4 GPU ile eğitiyoruz.


In [ ]:
import os
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'🚀 Model Eğitimi Başlatılıyor... Donanım: {device_arg} (GPU: {torch.cuda.is_available()})')

# YOLO11 Nano modelini yüklüyoruz
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,
    project='runs/train',
    name='yolo11n_uad_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## Adım 6: Model Başarım Testi ve Gerçek Fotoğraf Kıyaslaması 📸
Bağımsız test kümesi üzerinde kesinlik (mAP) değerlerini ölçer ve özel test fotoğrafı üzerinde eğitim öncesi vs. sonrası tespit farkını yan yana görselleştirir.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. Test Kümesi Değerlendirmesi
metrics = model.val(data='/content/ua_detrac_dataset/data.yaml')
print('\n' + '=' * 50)
print('      MODEL BAŞARI METRİKLERİ')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)

# 2. Gerçek Trafik Test Fotoğrafı Üzerinde Kıyaslama
custom_photo = '/content/federated-traffic-learning/test_samples/user_test_photo.jpg'
if not os.path.exists(custom_photo):
    custom_photo = 'test_samples/user_test_photo.jpg'

trained_weights = 'runs/train/yolo11n_uad_exp/weights/best.pt'

if os.path.exists(custom_photo) and os.path.exists(trained_weights):
    print('\n📸 Özel test fotoğrafı kıyaslaması yapılıyor...')
    base_model = YOLO('yolo11n.pt')
    base_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='base_pred', exist_ok=True)
    
    trained_model = YOLO(trained_weights)
    trained_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='trained_pred', exist_ok=True)
    
    base_imgs = glob.glob('runs/detect/base_pred/*.*')
    trained_imgs = glob.glob('runs/detect/trained_pred/*.*')
    
    if base_imgs and trained_imgs:
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(Image.open(base_imgs[0]))
        axes[0].set_title('Eğitim Öncesi: Base YOLO11n (Pretrained)', fontsize=15)
        axes[0].axis('off')
        
        axes[1].imshow(Image.open(trained_imgs[0]))
        axes[1].set_title('Eğitim Sonrası: Yeni Veriyle Eğitilmiş Model', fontsize=15)
        axes[1].axis('off')
        
        plt.tight_layout()
        out_comp = 'runs/train/yolo11n_uad_exp/custom_test_comparison.jpg'
        plt.savefig(out_comp, bbox_inches='tight', dpi=150)
        print(f'✅ Karşılaştırma görseli kaydedildi: {out_comp}')
        plt.show()


## Adım 7: Sonuçları Paketleme ve Otomatik İndirme 📦
Eğitilen `best.pt` model ağırlıklarını, grafiklerini ve karşılaştırma görselini tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [ ]:
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11n_uad_exp'
best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Eğitilmiş en başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('\n🎉 TEBRİKLER! Tüm eğitim ve test işlemleri başarıyla tamamlandı!')
print('👉 İnen "yolo11_results.zip" dosyasını masaüstündeki proje klasörünüze atıp asistanınıza bildirebilirsiniz.')
